# Coordinates and gridding

We map the same catalog in sky coordinates and in the pixel coordinates of its image, and show how SMPy projects
galaxy positions onto a plane, turns each shear into the frame of the map and bins galaxies into pixels. We then
choose a pixel size from the number of galaxies per pixel.

## Settings

Paths are relative to `docs/source/tutorials` in the SMPy repository.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

import smpy
from smpy.coordinates.radec import project

CATALOG = Path("../../../examples/data/forecast_lum_annular.fits")
SHEAR = dict(g1_col="g1_Rinv", g2_col="g2_Rinv", weight_col="weight")
SKY = dict(coord1_col="ra", coord2_col="dec", pixel_scale=0.4)  # arcminutes
IMAGE = dict(coord_system="pixel", coord1_col="X_IMAGE_se", coord2_col="Y_IMAGE_se", downsample_factor=170)

## Sky coordinates

SMPy maps RA/Dec catalogs on the plane tangent to the sky at the field center $(\alpha_0, \delta_0)$, the
gnomonic or TAN projection (Calabretta & Greisen 2002). A galaxy at $(\alpha, \delta)$ lands at

$$
x = -\frac{\cos\delta\,\sin(\alpha - \alpha_0)}{\cos c}, \qquad
y = \frac{\cos\delta_0 \sin\delta - \sin\delta_0 \cos\delta \cos(\alpha - \alpha_0)}{\cos c},
$$

where $\cos c = \sin\delta_0 \sin\delta + \cos\delta_0 \cos\delta \cos(\alpha - \alpha_0)$ and $c$ is its angular
distance from the center. The $x$ axis points west and $y$ north, so maps show east to the left, as the sky is seen.
The field center is the middle of the catalog's RA and Dec ranges.

In [ ]:
catalog = Table.read(CATALOG, hdu=1)
center = ((catalog["ra"].max() + catalog["ra"].min()) / 2, (catalog["dec"].max() + catalog["dec"].min()) / 2)
x, y = project(catalog["ra"], catalog["dec"], *center)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(60 * x, 60 * y, s=1)
ax.set_aspect("equal")
ax.set_xlabel("x, towards the west (arcmin)")
ax.set_ylabel("y, towards the north (arcmin)")
plt.show()

Each shear is read in its galaxy's own frame, with the first axis towards the west and the second towards the north.
Away from the center, north on the sky is turned from the map's $y$ axis by an angle
$\psi = \arctan[\sin(\alpha - \alpha_0) \sin\delta_0 / \cos(\alpha - \alpha_0)]$, so SMPy turns each shear by
$\psi$ before mapping. The angle grows with the width of the field; for this field of a few arcminutes it is
small.

In [ ]:
offset = np.deg2rad(catalog["ra"] - center[0])
psi = np.arctan2(np.sin(offset) * np.sin(np.deg2rad(center[1])), np.cos(offset))
print(f"Largest angle between north and the map's y axis: {np.rad2deg(np.abs(psi)).max():.3f} degrees")

## Pixels

The map is a grid of square pixels `pixel_scale` arcminutes wide, centered on the catalog and just large enough to
hold every galaxy. Each pixel holds the weighted mean shear of its galaxies. `MassMap.grid` keeps the binned shear,
the summed weights and the number of galaxies in each pixel.

In [ ]:
result = smpy.map_mass(CATALOG, **SHEAR, **SKY)
counts = result.grid.counts
print(f"{counts.shape[1]} x {counts.shape[0]} pixels holding {int(counts.sum())} of {len(catalog)} galaxies")
result.plot_counts()
plt.show()

## Choosing the pixel size

The shape noise of a pixel's mean shear falls as $\sigma_e / \sqrt{N}$ with its number of galaxies $N$, while larger
pixels blur the structures being mapped. Pixels without galaxies hold no shear: Kaiser-Squires and aperture mass
treat their shear as zero, and KS+ treats them as gaps to fill. A pixel size that leaves few pixels empty keeps
these effects small.

In [ ]:
for pixel_scale in (0.2, 0.4, 0.8):
    counts = smpy.map_mass(CATALOG, **SHEAR, **{**SKY, "pixel_scale": pixel_scale}).grid.counts
    print(f"{pixel_scale} arcmin: {counts.shape[1]} x {counts.shape[0]} pixels, "
          f"{counts.mean():.1f} galaxies per pixel, {np.mean(counts == 0):.0%} empty")

## Image coordinates

Catalogs can also be mapped in the pixel coordinates of their image, such as SExtractor's `X_IMAGE` and
`Y_IMAGE`. Each map pixel is then `downsample_factor` image pixels wide, and the shear is read along the image's $x$
and $y$ axes. Here 170 image pixels make map pixels of about the same width as the 0.4-arcminute pixels above.

In [ ]:
image = smpy.map_mass(CATALOG, **SHEAR, **IMAGE)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"wspace": 0.4})
result.plot("E", ax=axes[0], cmap="magma", plot_title="Sky coordinates")
image.plot("E", ax=axes[1], cmap="magma", plot_title="Image coordinates")
plt.show()

## Weights and invalid values

`weight_col` names a weight for each galaxy, such as its inverse shape-noise variance, and each pixel's shear is the
weighted mean of its galaxies. Galaxies with NaN, infinite or masked values must be removed first; SMPy stops with
an error that names the column.

In [ ]:
bad = catalog.copy()
bad["g1_Rinv"][0] = np.nan
try:
    smpy.map_mass(bad, **SHEAR, **SKY)
except ValueError as error:
    print(error)